# Insert initial arguments

In [ ]:
#Enter the following parameters
tcga_set = 'TCGA_breast'
gdc_file_name = 'gdc_sample_sheet.2022-06-16.tsv'
normal_tissue_type = 'normal gbm_brain tissue'
tumor_type = 'TCGA_breast'
tissue_type = 'TCGA_breast'

slide_file_location = f'{tcga_set}/slide.tsv'
gdc_file_location = f'{tcga_set}/{gdc_file_name}'
results_output_file = f'{tcga_set}/results'

# Import modules

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Reading the data

In [ ]:
#Cell composition data from TCGA BRCA slides

cols_to_use = ['case_id',
              'case_submitter_id',
              'sample_id',
              'sample_submitter_id',
              'percent_lymphocyte_infiltration',
              'percent_monocyte_infiltration',
              'percent_necrosis',
              'percent_neutrophil_infiltration',
              'percent_normal_cells',
              'percent_stromal_cells',
              'percent_tumor_cells',
              'percent_tumor_nuclei']


cell_composition = pd.read_csv(f'{slide_file_location}', sep = '\t', na_values = "'--", usecols = cols_to_use)
cell_composition.head(4)

In [ ]:
#Data gdc sample sheet information

gdc = pd.read_csv(f'{gdc_file_location}', sep = '\t')
gdc.head(4)

# Dictionaries

In [ ]:
#Dictionary with sample ID for keys and sample type as values

sample_id_sample_type = {}
for sample in gdc['Sample ID']:
  sample_type = gdc.loc[gdc['Sample ID'] == sample, 'Sample Type']
  sample_id_sample_type[sample] = sample_type.iloc[0]


# Merging cell composition and tumor types

In [ ]:
#Creating series with tissue type information for each sample

tissue_types = []
for i in cell_composition['sample_submitter_id']:
  if i in sample_id_sample_type.keys():
    single_sample_type = sample_id_sample_type[i]
    tissue_types.append(single_sample_type)
  else:
    tissue_types.append('unknown')
tissue_types = pd.Series(tissue_types)


In [ ]:
cell_composition['tissue_type'] = tissue_types
cell_composition.drop(cell_composition[cell_composition['tissue_type'] == 'unknown'].index, inplace = True)
cell_composition.head(4)

# Visualization

In [ ]:
normal_samples = cell_composition[cell_composition['tissue_type'] == 'Solid Tissue Normal']
tumor_samples = cell_composition[cell_composition['tissue_type'] != 'Solid Tissue Normal']

In [ ]:
df = tumor_samples[['percent_lymphocyte_infiltration',	'percent_monocyte_infiltration',	'percent_necrosis',	'percent_neutrophil_infiltration',	'percent_normal_cells',	'percent_stromal_cells',	'percent_tumor_cells']]
df = df.melt()
df.replace({'percent_lymphocyte_infiltration': 'lymphocytes',
            'percent_monocyte_infiltration' : 'monocytes',
            'percent_necrosis' : 'necrotic cells',
            'percent_neutrophil_infiltration' : 'neutrophils',
            'percent_normal_cells' : 'normal cells',
            'percent_stromal_cells' : 'stromal cells',
            'percent_tumor_cells' : 'tumor cells'}, inplace = True)
df.rename(columns = {'variable':'cell_type', 'value':'percentage'}, inplace = True)
df

In [ ]:
sns.boxplot(data = df, x = 'cell_type', y = 'percentage')
plt.xticks(rotation = 45, horizontalalignment = 'right')
plt.xlabel('Cell type')
plt.ylabel('Percentage of cell population')
plt.title(tissue_type)
plt.savefig(f'{results_output_file}/boxplot_{tissue_type}.svg', bbox_inches = 'tight')